https://platform.olimpiada-ai.ro/ro/problems/37

# setup + imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split

In [2]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, models
from PIL import Image

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

# SIZE = 224 # asa sunt alea de la resnet lowkey <- nvm o sa raman la 256 pt ca e close enough
SIZE = 256

cuda


In [4]:
# !unzip -q emoji_dataset.zip
# !ls

In [5]:
testcsv = pd.read_csv('test.csv')
traincsv = pd.read_csv('train.csv')

In [6]:
def load_images(data):
    imgs = np.zeros((len(data), SIZE, SIZE, 3), dtype=np.uint8)
    for i in range(len(data)):
        img = Image.open(data['Path'].iloc[i]).convert('RGB').resize((SIZE, SIZE))
        imgs[i] = np.array(img)
    return imgs

In [7]:
X = load_images(traincsv)
y = traincsv['Label']
y = torch.tensor(y.values, dtype=torch.float32)
X_test = load_images(testcsv)

print(X.shape, X_test.shape)

(1500, 256, 256, 3) (500, 256, 256, 3)


# preppyyyy

In [8]:
normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
def prep(xb):
    return normalize(xb.float() / 255)

def totensor(x):
  x = torch.from_numpy(x)
  x = x.permute(0, 3, 1, 2)
  return x

X = totensor(X)
X_test = totensor(X_test)

X = prep(X)
X_test = prep(X_test)

# validare

In [9]:
def customscore(pred, ref):
  total = 0
  for p, r in zip(pred, ref):
      d = abs(p - r)
      if d <= 1:
        total += 1
      elif d == 2:
        total += 0.75
      elif d == 3:
        total += 0.5
      elif d == 4:
         total += 0.25
  return total / len(ref) * 100

In [10]:
fit_rows, val_rows = train_test_split(range(len(X)), test_size=0.1, random_state=0)
X_val = X[val_rows]
y_val = y[val_rows]

In [11]:
lo = int(y.min())
hi = int(y.max())

# model

In [12]:
def predict(model, X, bsize=64):
  model.eval()
  preds = []
  with torch.no_grad():
      for s in range(0, len(X), bsize):
          xb = X[s:s+bsize].to(DEVICE)
          out = model(xb)[:, 0]
          for v in out:
              p = int(round(v.item()))
              if p < lo:
                  p = lo
              if p > hi:
                  p = hi
              preds.append(p)
  return preds

In [13]:


model = torchvision.models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
loss_fn = nn.L1Loss()

best_score = -1
for epochs in range(10):
  model.train()

  perm = np.random.permutation(fit_rows)
  total_loss = 0
  for s in range(0, len(fit_rows), 32):
    idx = perm[s:s+32]
    xb = X[idx].to(DEVICE)
    yb = y[idx].to(DEVICE)
    out = model(xb)[:, 0]
    loss = loss_fn(out, yb)
    opt.zero_grad();
    loss.backward();
    opt.step();
    total_loss += loss.item() * len(idx)
  score = customscore(predict(model, X_val), y_val)
  print(epochs, round(total_loss / len(perm), 4), round(score, 2))
  if score > best_score:
    best_score = score
    torch.save(model.state_dict(), "best.pt")



0 1.3864 98.83
1 0.4945 100.0
2 0.3616 100.0
3 0.2987 100.0
4 0.2773 100.0
5 0.2236 100.0
6 0.2378 100.0
7 0.215 100.0
8 0.211 100.0
9 0.1827 100.0


# pred + submisie

In [14]:
# model.load_state_dict(torch.load("best.pt"))
preds = predict(model, X_test)

answers = pd.DataFrame({
    'SampleID': testcsv['SampleID'],
    'PredictedLabel': preds
})

answers.to_csv('submission.csv', index=False)